# Download NOAA ERSST v5 Sea Surface Temperature Data

Downloads monthly SST data from NOAA's [Extended Reconstructed Sea Surface
Temperature (ERSST) Version 5](https://www.ncei.noaa.gov/products/extended-reconstructed-sst) dataset.

ERSST v5 specifications:
- Grid: 2° × 2° (89 lat × 180 lon)
- Coverage: 1854-present
- Variables: SST (sea surface temperature)
- Format: NetCDF (CF conventions)

Each file is ~200KB, full dataset ~400MB.

**Citation:** Huang et al. (2017): NOAA Extended Reconstructed Sea Surface Temperature
(ERSST), Version 5. doi:10.7289/V5T72FNM

**Requirements:** `pip install requests`

In [ ]:
# Configuration - Modify these to experiment

START_YEAR = 1980   # First year to download
END_YEAR = 2023     # Last year to download

# For full analysis, use START_YEAR = 1854

In [ ]:
import os
import requests
from pathlib import Path

BASE_URL = "https://www.ncei.noaa.gov/pub/data/cmb/ersst/v5/netcdf/"
DATA_DIR = Path("data")


def download_ersst_year(year):
    """Download all monthly ERSST files for a given year."""
    DATA_DIR.mkdir(parents=True, exist_ok=True)

    files_downloaded = []

    for month in range(1, 13):
        filename = f"ersst.v5.{year:04d}{month:02d}.nc"
        url = BASE_URL + filename
        outpath = DATA_DIR / filename

        if outpath.exists():
            print(f"  {filename} already exists, skipping")
            files_downloaded.append(str(outpath))
            continue

        print(f"  Downloading {filename}...", end="", flush=True)
        try:
            response = requests.get(url, timeout=60)
            response.raise_for_status()
            with open(outpath, 'wb') as f:
                f.write(response.content)
            print(" done")
            files_downloaded.append(str(outpath))
        except Exception as e:
            print(f" failed ({type(e).__name__})")

    return files_downloaded

## Download Data

In [ ]:
print(f"Downloading NOAA ERSST v5 data")
print(f"  Years: {START_YEAR} - {END_YEAR}")
print(f"  Source: {BASE_URL}")
print(f"  This will take a few minutes...")
print()

all_files = []

for year in range(START_YEAR, END_YEAR + 1):
    print(f"Year {year}:")
    files = download_ersst_year(year)
    all_files.extend(files)

print(f"\nDownload complete: {len(all_files)} files")

## List Local Files

In [ ]:
if DATA_DIR.is_dir():
    local_files = sorted([str(f) for f in DATA_DIR.glob("*.nc")])
    print(f"Local files: {len(local_files)}")
    if local_files:
        print(f"  First: {os.path.basename(local_files[0])}")
        print(f"  Last:  {os.path.basename(local_files[-1])}")
else:
    print("No data directory found.")

print(f"\nTo download more years, change START_YEAR/END_YEAR above and re-run.")